# MachineInnovators — review del progetto MLOps

Repository: [social-sentiment-mlops](https://github.com/flaviotorre-dev/social-sentiment-mlops)  
Immagine pubblica: [GitHub Container Registry](https://github.com/flaviotorre-dev/social-sentiment-mlops/pkgs/container/social-sentiment-mlops)

Progetto didattico: espone un classificatore pre-addestrato per sentiment positive, neutral e negative. Questo notebook documenta le scelte e i limiti; non esegue training.

## Architettura

```text
Client --HTTP--> FastAPI (/predict, /health)
                     | modello caricato all'avvio
                     v
       CardiffNLP Twitter RoBERTa sentiment

Prometheus --scrapes /metrics--> FastAPI
Grafana --queries-------------> Prometheus

GitHub Actions --test/build--> GHCR
Airflow DAG settimanale: collect_data -> evaluate_model -> decide -> retrain | skip
```

L'API, Prometheus e Grafana sono avviabili con Docker Compose. Il DAG è un file dimostrativo e non richiede che Airflow faccia parte dello stack applicativo.

## API: contratto e comportamento

L'interfaccia HTTP è intenzionalmente piccola e stateless: il client invia un testo e riceve una singola classificazione; non ci sono sessioni o stato applicativo da sincronizzare.

| Metodo e percorso | Input/output | Scelte e comportamento |
|---|---|---|
| `GET /health` | `{"status":"ok"}` | Controllo basilare di processo. Non verifica separatamente che il modello sia pronto o che Prometheus sia raggiungibile. |
| `POST /predict` | JSON `{"text":"..."}`; risposta `{"label":"positive|neutral|negative","score":0..1}` | Pydantic rifiuta testo vuoto o oltre 512 caratteri con HTTP 422. Il modello è caricato una volta durante il lifespan e riusato: si evita il costo del caricamento ad ogni richiesta, al prezzo del consumo di RAM per processo. |
| `GET /metrics` | Formato Prometheus | Endpoint tecnico per scraping, non destinato ai client finali; al momento non è protetto da autenticazione. |

La risposta di errore di validazione è quella standard FastAPI/Pydantic (422). Gli errori interni del modello diventano errori server e il middleware registra lo status 500; non è prevista una risposta di ripiego che sembri una predizione valida. FastAPI espone inoltre la documentazione OpenAPI interattiva sui percorsi predefiniti `/docs` e `/openapi.json`.

La lunghezza massima è espressa in caratteri, mentre il modello tokenizza e tronca a 512 token. I due limiti non sono equivalenti: 512 caratteri normalmente producono meno di 512 token, ma la scelta evita payload arbitrariamente grandi senza promettere un numero preciso di token. Il punteggio restituito è quello del classificatore, non una confidenza calibrata. Prima di fissare il contratto per un cliente andrebbero concordati lingua, normalizzazione, limiti di payload, timeout e comportamento in caso di indisponibilità del modello.

## Docker: immagine e stack locale

Il `Dockerfile` parte da `python:3.14-slim`, installa le dipendenze Python fissate in `requirements.txt`, scarica il modello durante la build e solo dopo copia `app/`. Questa sequenza sfrutta la cache dei layer: le modifiche al codice non obbligano a reinstallare dipendenze e riscaricare il modello. Il container avvia un singolo processo Uvicorn sulla porta 8000; non include proxy TLS né autenticazione.

**Perché includere il modello nell'immagine:** l'avvio non dipende dalla rete o dall'accesso a Hugging Face e ogni container usa un artefatto già presente. Il compromesso è un'immagine molto grande (circa 3,82 GB nel contesto misurato) e una build/push più lenti. Un volume o download all'avvio ridurrebbe la dimensione distribuita ma introdurrebbe dipendenza dalla rete, gestione della cache e rischi di versioni non riproducibili; ONNX o un modello distillato richiederebbero benchmark e una verifica di qualità prima della sostituzione.

Docker Compose avvia quattro componenti: API (porta host 8000), Prometheus (9090), Grafana (3000) e i due strumenti di monitoraggio. Prometheus interroga `api:8000` ogni 15 secondi sulla rete Compose; Grafana usa il datasource provisionato che punta a `http://prometheus:9090`. I volumi nominati conservano i dati di Prometheus e Grafana attraverso la ricreazione dei container, ma non sono un backup né una strategia di disaster recovery. La password amministrativa Grafana è richiesta tramite `GRAFANA_ADMIN_PASSWORD`; non va inserita nel repository.

Il mapping porte è adatto alla prova locale, ma espone i servizi sulle interfacce host secondo la configurazione Docker. Per un cliente reale andrebbero pubblicati solo gli endpoint necessari, collocati gli strumenti interni su una rete amministrativa e messi TLS e autenticazione davanti all'API e alle console. `depends_on` ordina l'avvio, ma non garantisce che i servizi siano già pronti: mancano health check Compose e una strategia di riavvio/alta disponibilità. Il Compose non è quindi un installer di produzione.

Alternative come Kubernetes, un proxy inverso, immagini multi-stage e un servizio di model serving separato non sono incluse perché aggiungerebbero componenti e complessità non necessari per dimostrare il flusso. Per un'installazione cliente si sceglierebbero solo dopo aver raccolto requisiti di disponibilità, rete, carico e gestione operativa.

## Perché queste scelte

- **FastAPI**: API HTTP piccola, validazione della richiesta e documentazione OpenAPI automatica.
- **Modello pre-addestrato CardiffNLP**: evita di inventare un dataset o un training per coprire il caso d'uso didattico.
- **Python 3.14**: versione fissata e usata coerentemente in ambiente, Docker e CI.
- **PyTorch CPU-only**: non serve una GPU per il demo e si evita la dipendenza CUDA; l'immagine resta comunque grande (circa 3,82 GB).
- **Due job CI**: il job di build dipende dal successo di lint e test; non si pubblica un'immagine se i controlli falliscono.
- **GHCR**: la pubblicazione usa il `GITHUB_TOKEN` e non richiede un token di registry aggiuntivo.

Alternative non adottate: un modello distillato o ONNX per ridurre dimensioni e latenza; una build Docker multi-stage; un deploy su Hugging Face Spaces. Quest'ultimo richiederebbe uno Space Docker, un `HF_TOKEN` conservato nei secret del repository e un passaggio di deploy dedicato.

## CI/CD

La GitHub Action esegue `ruff check`, `ruff format --check` e pytest con Python 3.14 e cache del modello Hugging Face. Se il job `test` riesce, un push su `main` costruisce l'immagine e la pubblica su GHCR con tag `latest` e tag SHA. Pull request e push eseguono i controlli; il job di pubblicazione è riservato ai push su `main`.

La pipeline verifica qualità e pubblicazione dell'immagine, ma non effettua il deploy dell'applicazione su un ambiente di produzione.

## Monitoraggio

Prometheus raccoglie `/metrics` ogni 15 secondi; Grafana carica datasource e dashboard tramite provisioning. I segnali sono:

- volume di richieste per endpoint;
- latenza, con visualizzazione del p95;
- richieste con status 4xx/5xx;
- distribuzione delle classi predette;
- lunghezza dei testi in caratteri.

L'unico alert didattico è **quota di predizioni negative > 60%** sulle predizioni degli ultimi 15 minuti, mantenuta per 5 minuti. È una soglia semplice per segnalare uno spostamento degli output, non una soglia calibrata: per un servizio reale andrebbe definita con dati storici e tolleranza ai cambiamenti normali di traffico. L'alert è visibile in Prometheus, ma non sono configurate notifiche esterne.

## Data drift e concept drift

**Data drift** significa che cambia la distribuzione degli input, ad esempio lessico, lunghezza o argomenti dei testi. Può essere osservato anche senza conoscere il sentiment corretto, confrontando le caratteristiche degli input nel tempo.

**Concept drift** significa che cambia la relazione tra testo e sentiment (l'etichetta corretta). La sola distribuzione degli input o delle predizioni non basta a confermarlo: servono etichette reali su un campione di testi e una misura periodica degli errori del modello. Le metriche attuali descrivono traffico e output, non misurano l'accuracy online.

## Logica del DAG

Il DAG `weekly_sentiment_retraining` è pianificato con `@weekly`:

1. `collect_data` restituisce un conteggio dimostrativo dei campioni via XCom.
2. `evaluate_model` legge quel valore da XCom e restituisce un'accuracy stub.
3. `decide` legge l'accuracy e confronta con la soglia 0,75.
4. Se l'accuracy è sotto soglia, il branch esegue `retrain`; altrimenti `skip`.

Nel codice demo l'accuracy stub è 0,72, quindi il percorso scelto è `retrain`. Nessun dato viene realmente raccolto, nessun modello viene valutato e il task `retrain` non addestra un modello.

## Limiti attuali

- Il modello non è stato fine-tuned sui dati di un cliente e le classi non sono una misura di verità.
- Non ci sono autenticazione, rate limiting, gestione di dati sensibili o obiettivi SLO.
- Il servizio carica un solo modello in memoria e non è dimensionato per alta disponibilità.
- Il conteggio per classe non permette di calcolare accuracy senza etichette reali.
- L'immagine Docker è pesante; il compose è per una demo locale, senza backup, TLS o notifiche degli alert.
- I task Airflow sono stub e non esiste una pipeline di training o validazione reale.

## Verifica puntuale del repository

Alcuni limiti si vedono direttamente nell'implementazione e chiariscono perché il compose va considerato locale/dimostrativo:

- **Esposizione di rete**: Compose pubblica le porte di API, Prometheus e Grafana sull'host. La password configurata protegge Grafana, ma API e Prometheus non hanno autenticazione; non c'è TLS. È comodo per provare il sistema, non è una configurazione da esporre a una rete aziendale o a Internet.
- **Avvio e disponibilità del modello**: l'API carica il modello nel lifespan prima di servire richieste. L'immagine Docker lo scarica durante la build, mentre l'avvio locale può richiedere la cache o l'accesso a Hugging Face. Se il caricamento fallisce, il servizio non parte. `/health` risponde con uno stato fisso: è utile come controllo basilare, ma non offre controlli distinti di liveness e readiness né diagnostica.
- **Riproducibilità**: `MODEL_NAME` identifica il repository del modello, non una revisione immutabile. Una build futura potrebbe quindi scaricare pesi diversi. Per riprodurre e fare rollback servono revisione esatta, checksum e conservazione dell'artefatto usato.
- **Test e CI**: i test di modello eseguono davvero la pipeline Transformers e dipendono dal modello disponibile in cache o scaricabile. Sono utili come test d'integrazione, ma possono essere lenti e dipendere da un servizio esterno; test unitari con pipeline simulata e test d'integrazione separati e versionati renderebbero più chiara questa distinzione.
- **Container e scalabilità**: il Dockerfile non dichiara un utente non privilegiato. Inoltre ogni processo/replica che carica il modello consuma memoria per i propri pesi; il progetto non misura capacità, concorrenza o tempi sotto carico. Prima di aumentare i worker o le repliche occorre misurare CPU, RAM e latenza.
- **Significato del punteggio**: `score` è la confidenza restituita dal modello, non una probabilità calibrata né una garanzia di correttezza. Va validata su dati rappresentativi del cliente prima di usarla per decisioni automatiche.
- **Alert**: la quota negativa è un segnale esplorativo basato sulle sole predizioni. Con pochi esempi può oscillare molto e non dimostra un peggioramento dell'accuracy; in produzione la soglia dovrebbe includere un volume minimo e notifiche a un canale presidiato.

Questi punti completano la lista dei limiti: indicano cosa verificare nel codice e quale rischio operativo deriva da ciascuna scelta.

## Cosa servirebbe per un cliente reale

Prima del rilascio servirebbero requisiti e dataset rappresentativi, valutazione per segmento e lingua, revisione di bias e privacy, autorizzazione e protezione degli endpoint, logging conforme alle policy, deployment con health check e rollback, risorse dimensionate, alert con un canale di notifica, metriche di qualità basate su etichette e un processo di approvazione per ogni nuovo modello. I dati degli utenti andrebbero minimizzati e conservati secondo una policy concordata.

## Cosa farei con più tempo

Misurerei latenza e memoria con test di carico, confronterei il modello attuale con una variante più piccola, fisserei e verificherei tutte le immagini di base tramite digest, aggiungerei test di integrazione del compose e controlli sul formato delle metriche, e definirei una valutazione offline con campioni etichettati. Solo dopo aver stabilito requisiti di qualità e operativi aggiungerei retraining reale, model registry e deploy controllato.